# LLM Benchmark — GGUF Runner (llama.cpp)

GGUF modeller için. vLLM yerine llama-cpp-python server kullanır.
Aynı benchmark sistemi, aynı test setleri.

## Kullanım
1. A bölümünü bir kere çalıştır
2. B1'de model URL ve dosya adını değiştir, B2'den itibaren çalıştır

---
# A) İlk Kurulum (bir kere)

In [ ]:
# HuggingFace login
import os
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    !huggingface-cli login --token {hf_token} --add-to-git-credential
    print('✅ HuggingFace login başarılı')
except Exception as e:
    print(f'⚠️ HF_TOKEN bulunamadı: {e}')

!pip -q install llama-cpp-python[server] httpx pydantic hypothesis nest_asyncio huggingface_hub
!nvidia-smi

In [ ]:
import os, sys

ROOT = '/content/llm'
LOG_DIR = f'{ROOT}/logs'
CACHE_DIR = f'{ROOT}/cache'
MODEL_DIR = f'{ROOT}/models'
PROJECT_DIR = f'{ROOT}/benchmark'

for p in [ROOT, LOG_DIR, CACHE_DIR, MODEL_DIR]:
    os.makedirs(p, exist_ok=True)

REPO_URL = 'https://github.com/orhan-kaplan/benchmark.git'
if os.path.exists(PROJECT_DIR):
    print('📦 Repo mevcut, güncelleniyor...')
    !git -C {PROJECT_DIR} checkout -- benchmark_data/models.json 2>/dev/null
    !git -C {PROJECT_DIR} pull --ff-only
else:
    print('📦 Repo klonlanıyor...')
    !git clone {REPO_URL} {PROJECT_DIR}

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print(f'✅ Proje hazır: {PROJECT_DIR}')

---
# B) Model Test Döngüsü

In [ ]:
import json

#  ╔══════════════════════════════════════════════════════════════╗
#  ║  GGUF MODEL SEÇ — ID veya isim yaz                        ║
#  ╚══════════════════════════════════════════════════════════════╝
MODEL = 1  # ID veya isim: 1, 'qwen3.5-35b-uncensored-q4' vs.

# JSON'dan config oku
with open(f'{PROJECT_DIR}/benchmark_data/models-llama.json') as f:
    models_data = json.load(f)['models']

m = None
if isinstance(MODEL, int):
    m = next((x for x in models_data if x.get('id') == MODEL), None)
elif isinstance(MODEL, str):
    m = next((x for x in models_data if x['name'] == MODEL), None)

if m is None:
    print(f'❌ Model bulunamadı: {MODEL}')
    for x in models_data:
        print(f"  {x.get('id', '?'):2}. {x['name']}")
    raise ValueError(f'Geçersiz model: {MODEL}')

HF_REPO = m['hf_repo']
GGUF_FILE = m['gguf_file']
MODEL_LABEL = m['name']
N_GPU_LAYERS = m.get('llama', {}).get('n_gpu_layers', -1)
N_CTX = m.get('llama', {}).get('n_ctx', 8192)
TEST_SETS = m.get('benchmark', {}).get('test_sets', ['coding'])
TEMPERATURE = m.get('benchmark', {}).get('temperature', 0.7)
MAX_TOKENS = m.get('benchmark', {}).get('max_tokens', 1024)
TOP_P = m.get('benchmark', {}).get('top_p', 1.0)
PORT = 8090
VLLM_BASE_URL = f'http://localhost:{PORT}'
MODEL_PATH = f'{MODEL_DIR}/{GGUF_FILE}'

print(f'[{m.get("id")}] {MODEL_LABEL}')
print(f'    Repo: {HF_REPO}')
print(f'    File: {GGUF_FILE}')
print(f'    Test: {TEST_SETS}')

In [ ]:
import subprocess, time, requests

# Önceki server'ı durdur
print('🛑 Önceki server durduruluyor...')
subprocess.run('pkill -f llama_cpp.server || true', shell=True, check=False)
time.sleep(2)

# Model indir (yoksa)
if not os.path.exists(MODEL_PATH):
    print(f'📥 Model indiriliyor: {GGUF_FILE}')
    from huggingface_hub import hf_hub_download
    hf_hub_download(repo_id=HF_REPO, filename=GGUF_FILE, local_dir=MODEL_DIR)
    print('✅ İndirildi')
else:
    print(f'✅ Model zaten mevcut: {MODEL_PATH}')

# llama.cpp server başlat
LOG_PATH = f'{LOG_DIR}/llama-{MODEL_LABEL}.log'
cmd = [
    'python', '-m', 'llama_cpp.server',
    '--model', MODEL_PATH,
    '--n_gpu_layers', str(N_GPU_LAYERS),
    '--n_ctx', str(N_CTX),
    '--host', '0.0.0.0',
    '--port', str(PORT),
]

with open(LOG_PATH, 'w') as f:
    proc = subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL)

print(f'🚀 Server başlatıldı (PID: {proc.pid})')

# Bekle
start_time = time.time()
server_ready = False
while time.time() - start_time < 300:
    try:
        if requests.get(f'{VLLM_BASE_URL}/v1/models', timeout=3).status_code == 200:
            server_ready = True
            break
    except: pass
    if proc.poll() is not None:
        print(f'❌ Server çöktü! (exit: {proc.returncode})')
        with open(LOG_PATH) as f: print(f.read()[-1000:])
        break
    print(f'[{int(time.time()-start_time):3d}s] ⏳ Başlatılıyor')
    time.sleep(5)

if server_ready:
    print(f'\n✅ Server hazır! ({int(time.time()-start_time)}s)')
    r = requests.post(f'{VLLM_BASE_URL}/v1/chat/completions',
        json={'model': MODEL_PATH, 'messages': [{'role':'user','content':'Say hello'}], 'max_tokens': 32}, timeout=60)
    if r.status_code == 200:
        msg = r.json()['choices'][0]['message']
        txt = msg.get('content') or msg.get('reasoning') or 'No content'
        print(f'   💬 {txt[:150]}')

In [ ]:
import asyncio, importlib
from pathlib import Path

import benchmark.runner, benchmark.catalog, benchmark.test_sets
import benchmark.api_client, benchmark.metrics, benchmark.storage, benchmark.models
for mod in [benchmark.models, benchmark.storage, benchmark.metrics,
            benchmark.api_client, benchmark.catalog, benchmark.test_sets, benchmark.runner]:
    importlib.reload(mod)

from benchmark.runner import BenchmarkRunner
from benchmark.catalog import ModelCatalog
from benchmark.test_sets import TestSetManager
from benchmark.api_client import APIClient
from benchmark.metrics import MetricsCollector, VRAMTracker
from benchmark.storage import StorageManager
from benchmark.models import BenchmarkRunConfig, GenerationParams, ModelEntry, Backend

data_path = Path(PROJECT_DIR) / 'benchmark_data'
storage = StorageManager(data_path)
catalog = ModelCatalog(data_path)
test_sets = TestSetManager(data_path)
api_client = APIClient(timeout=300.0, max_retries=2)
metrics_collector = MetricsCollector()
vram_tracker = VRAMTracker()

# Model endpoint güncelle — llama.cpp model adı olarak dosya yolunu kullanır
if catalog.get(MODEL_LABEL) is None:
    catalog.register(ModelEntry(name=MODEL_LABEL, repo=MODEL_PATH, format='GGUF',
                               backend=Backend.LLAMA_CPP, tags=['gguf'], api_endpoint=VLLM_BASE_URL))
    print(f'Model eklendi: {MODEL_LABEL}')
else:
    catalog.update(MODEL_LABEL, {'api_endpoint': VLLM_BASE_URL, 'repo': MODEL_PATH})
    print(f'Endpoint güncellendi: {MODEL_LABEL}')

runner = BenchmarkRunner(catalog=catalog, test_set_manager=test_sets, api_client=api_client,
                         metrics_collector=metrics_collector, storage=storage, vram_tracker=vram_tracker)

async def run_benchmarks():
    ids = []
    for ts_name in TEST_SETS:
        print(f"\n{'='*60}")
        print(f'Benchmark: {ts_name} × {MODEL_LABEL}')
        print(f"{'='*60}")
        config = BenchmarkRunConfig(test_set_name=ts_name, model_names=[MODEL_LABEL],
            params=GenerationParams(temperature=TEMPERATURE, max_tokens=MAX_TOKENS, top_p=TOP_P))
        result = await runner.run(config)
        ids.append(result.run_id)
        ok = sum(1 for r in result.results if r.success)
        fail = sum(1 for r in result.results if not r.success)
        print(f'✅ run_id={result.run_id} | Başarılı: {ok}, Başarısız: {fail}')
    await api_client.close()
    return ids

try:
    loop = asyncio.get_running_loop()
    import nest_asyncio; nest_asyncio.apply()
    run_ids = asyncio.run(run_benchmarks())
except RuntimeError:
    run_ids = asyncio.run(run_benchmarks())

print(f"\n🏁 Tamamlandı. Run ID'ler: {run_ids}")

In [ ]:
from benchmark.reporter import ReportGenerator
reporter = ReportGenerator(storage=storage)

for run_id in run_ids:
    print(f"\n{'='*60}")
    print(f'Rapor: {run_id}')
    print(f"{'='*60}")
    summary = reporter.generate_summary(run_id)
    print(f'  Prompt: {summary.total_prompts} | Model: {summary.total_models} | ✅ {summary.successful_results} | ❌ {summary.failed_results}')
    print()
    results = storage.read_results(run_id)
    for r in results:
        s = '✅' if r.get('success') else '❌'
        m = r.get('metrics') or {}
        t = m.get('total_time_ms') or 0
        tps = m.get('tokens_per_second') or 0
        tok = m.get('completion_tokens') or 0
        print(f"  {s} {r['prompt_id']:15s} | {t:7.0f}ms | {tps:6.1f} tok/s | {tok} tok")
        if r.get('error'): print(f"     ⚠️ {r['error'][:100]}")
    reporter.export_json(run_id, storage.runs_dir / run_id / 'report.json')
    reporter.export_csv(run_id, storage.runs_dir / run_id / 'report.csv')

In [ ]:
INSPECT_RUN = run_ids[0] if run_ids else ''
INSPECT_PROMPT = None

if INSPECT_RUN:
    for r in storage.read_results(INSPECT_RUN):
        if INSPECT_PROMPT and r['prompt_id'] != INSPECT_PROMPT: continue
        print(f"\n{'─'*60}")
        print(f"Prompt: {r['prompt_id']} | Model: {r['model_name']}")
        print(f"{'─'*60}")
        print(r.get('response_text') or r.get('error') or 'Yanıt yok')

In [ ]:
import shutil, json
from google.colab import files

for run_id in run_ids:
    run_dir = str(storage.runs_dir / run_id)
    with open(f'{run_dir}/meta.json') as f:
        ts_name = json.load(f).get('config', {}).get('test_set_name', 'unknown')
    zip_name = f'{MODEL_LABEL}_{ts_name}_{run_id}'
    zip_path = f'/content/{zip_name}'
    shutil.make_archive(zip_path, 'zip', run_dir)
    print(f'📦 {zip_name}.zip')
    files.download(f'{zip_path}.zip')

---
# C) Opsiyonel

In [ ]:
NGROK_ENABLED = False
NGROK_AUTHTOKEN = 'TOKEN'

if NGROK_ENABLED:
    try:
        from google.colab import userdata
        NGROK_AUTHTOKEN = userdata.get('NGROK_AUTHTOKEN')
    except: pass
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    ngrok.kill()
    tunnel = ngrok.connect(PORT, 'http')
    print(f'🌐 {tunnel.public_url}')
    print(f'Open WebUI → {tunnel.public_url}/v1')
else:
    print('ngrok devre dışı')

In [ ]:
import time, requests
print('Canlı tutma aktif. Durdurmak için interrupt et.')
while True:
    try: s = '✅' if requests.get(f'{VLLM_BASE_URL}/v1/models', timeout=5).status_code == 200 else '⚠️'
    except: s = '❌'
    print(f'{s} {time.strftime("%H:%M:%S")}')
    time.sleep(30)

---
### Yardımcı
```python
# Eski model sil (disk alanı)
!rm -rf /content/llm/models/*

# Log kontrol
!cat /content/llm/logs/llama-*.log | tail -50

# GPU kullanımı
!nvidia-smi

# Server durdur
!pkill -f llama_cpp.server
```